# Einen Hook schreiben

Ein PreToolUse-Hook sieht jeden Tool-Aufruf, bevor Claude Code ihn ausführt, und kann ihn blockieren. Hier schreibt ihr die Prüfung dahinter als Python-Funktion: Sie bekommt einen Aufruf und antwortet „erlauben“ oder „blockieren“, mit Grund. Ziel: Der Agent kommt nicht an die `.env` heran, an die `.env.example` schon.

1. So sieht ein Aufruf aus
2. Die `.env` schützen, auch in der Shell
3. Umwege suchen und schliessen
4. Aus der Funktion einen echten Hook machen

### Aufgaben

**Für alle** (rund 10 Minuten)

1. Den Test der Startfunktion lesen: Welche Aufrufe lässt sie durch? *(Einstieg)*
2. Erster Versuch: blockieren, wenn `.env` im Befehl steht
3. Den Fehlalarm beheben, bis 13 von 13 stimmen

**Bonus**

4. Einen Aufruf finden, den euer Hook durchlässt
5. Knobelaufgabe: zwei Umwege schliessen, ohne dass einer der 13 Fälle kippt

Zeitbox 15 Minuten, zu zweit. Jede Aufgabe steht noch einmal an der Stelle, an der ihr sie macht. Abschnitt 4 ist zum Ansehen.

## 1. So sieht ein Aufruf aus

Claude Code gibt dem Hook den Aufruf als JSON auf stdin. Wichtig sind zwei Felder: `tool_name` sagt, welches Tool, `tool_input`, womit. Bei Read, Edit und Write steht dort der absolute Pfad `file_path`, bei Bash der Befehl `command`. Zwei Aufrufe im Projekt `kurs-service`:

In [ ]:
# Nur in Colab: Hilfsdateien und Daten aus dem Repo holen. Lokal tut die Zelle nichts.
import os
import subprocess
import sys

if "google.colab" in sys.modules:
    repo = "/content/ki-workshop-handson"
    if not os.path.exists(repo):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/dprinz/ki-workshop-handson.git", repo], check=True)
    os.chdir(f"{repo}/09-hook-schreiben")

In [1]:
import json

from helfer import FAELLE_ENV, FAELLE_UMWEGE, bash, edit, grep, kurz, probiere, read, teste, write

print(json.dumps(read(".env"), indent=2))
print(json.dumps(bash("cat .env"), indent=2))

{
  "hook_event_name": "PreToolUse",
  "cwd": "/home/dev/kurs-service",
  "tool_name": "Read",
  "tool_input": {
    "file_path": "/home/dev/kurs-service/.env"
  }
}
{
  "hook_event_name": "PreToolUse",
  "cwd": "/home/dev/kurs-service",
  "tool_name": "Bash",
  "tool_input": {
    "command": "cat .env"
  }
}


In Python ist das ein `dict`, also `aufruf["tool_name"]` und `aufruf["tool_input"]["command"]`. Echte Aufrufe haben noch ein paar Felder mehr, etwa `session_id` und `transcript_path`. Die brauchen wir hier nicht.

## 2. Für alle: die `.env` schützen, auch in der Shell

Die Funktion unten macht dasselbe wie der Hook von AstroPlanner auf der Folie: Sie schaut nur auf `file_path`. Der Test darunter schickt ihr 13 Aufrufe und vergleicht ihre Antwort mit der Spalte „Soll“.

> **Aufgabe 1** (Einstieg): Die Zelle ausführen (Shift+Enter) und die Tabelle lesen. 7 von 13 stimmen. Welche Zeilen sind FALSCH, und was haben sie gemeinsam?

<details><summary>Antwort</summary>

Alle sechs sind Bash-Aufrufe. Ein Bash-Aufruf hat kein `file_path`, also lässt die Funktion jeden Shell-Befehl durch, auch `echo … >> .env`.

</details>

> **Aufgabe 2**: Im Zweig `if werkzeug == "Bash":` blockieren, wenn im Befehl der Text `.env` vorkommt. Zelle ändern und neu ausführen, der Test läuft gleich mit. Fertig, wenn nur noch eine Zeile FALSCH ist. Welche, und warum?

<details><summary>Tipp 1</summary>

`".env" in befehl` ist wahr, wenn der Text im Befehl vorkommt, wie `befehl.contains(".env")` in Java. Die Antwort ist ein Paar wie im Zweig für Read, Edit und Write: `return "blockieren", "Grund"`. Der Grund geht im echten Hook an den Agent; er darf also kurz sagen, was stattdessen geht.

</details>
<details><summary>Tipp 2</summary>

Der Zweig für Bash, eingerückt wie in der Zelle:

```python
    if werkzeug == "Bash":
        befehl = eingabe["command"]
        if ".env" in befehl:
            return "blockieren", "Der Befehl greift auf die .env zu. Vorlage: .env.example"
```

</details>

In [2]:
import re
from pathlib import PurePosixPath


def pruefe(aufruf):
    werkzeug = aufruf["tool_name"]
    eingabe = aufruf["tool_input"]

    if werkzeug in ("Read", "Edit", "Write"):
        if PurePosixPath(eingabe["file_path"]).name == ".env":
            return "blockieren", "Die .env enthält Secrets. Vorlage: .env.example"

    if werkzeug == "Bash":
        befehl = eingabe["command"]
        # Hier fehlt die Prüfung des Befehls.

    return "erlauben", ""


teste(pruefe, FAELLE_ENV)

       Nr  Soll        Ist         Tool   Aufruf
        1  blockieren  blockieren  Read   .env
        2  erlauben    erlauben    Read   .env.example
        3  erlauben    erlauben    Read   README.md
        4  erlauben    erlauben    Edit   .env.example
        5  blockieren  blockieren  Write  .env
FALSCH  6  blockieren  erlauben    Bash   cat .env
        7  erlauben    erlauben    Bash   cat .env.example
FALSCH  8  blockieren  erlauben    Bash   echo "DB_PASSWORD=geheim" >> .env
FALSCH  9  blockieren  erlauben    Bash   cp .env.example .env
FALSCH 10  blockieren  erlauben    Bash   grep TOKEN .env
FALSCH 11  blockieren  erlauben    Bash   cp .env /tmp/env-kopie
FALSCH 12  blockieren  erlauben    Bash   python3 -c "print(open('.env').read())"
       13  erlauben    erlauben    Bash   mvn spring-boot:run

7 von 13 wie erwartet. Falsch: 6, 8, 9, 10, 11, 12


Mit dem ersten Versuch ist Nr. 7, `cat .env.example`, ein Fehlalarm: Auch dort steht der Text `.env`.

> **Aufgabe 3**: Die Prüfung in derselben Zelle so verfeinern, dass `.env.example` wieder durchgeht und alles andere blockiert bleibt. Fertig, wenn unter der Tabelle „✓ 13 von 13 wie erwartet.“ steht.

<details><summary>Tipp 1</summary>

Schaut, was in den Befehlen direkt nach `.env` kommt: bei `cat .env.example` ein Punkt, bei den anderen ein Leerzeichen, ein `'` oder das Ende des Befehls.

</details>
<details><summary>Tipp 2</summary>

Zwei Wege, beide schaffen 13 von 13:

- `.env.example` vor der Prüfung aus dem Befehl streichen: `if ".env" in befehl.replace(".env.example", ""):`
- ein regulärer Ausdruck: `if re.search(r"\.env(?![\w.-])", befehl):`. `(?![\w.-])` heisst: Danach folgt kein Buchstabe, keine Ziffer, kein Punkt und kein Bindestrich. `re.search` sucht wie `Matcher.find()` in Java, `re` ist in der Zelle schon importiert.

</details>

In [3]:
# @title Lösung (aufklappen)
import re
from pathlib import PurePosixPath

# .env als eigenes Wort: davor und danach kein Buchstabe, keine Ziffer, kein Punkt, kein Bindestrich.
# So passen .env, './.env' und /pfad/.env, aber nicht .env.example oder .envrc.
ENV_IM_BEFEHL = re.compile(r"(?<![\w.-])\.env(?![\w.-])")


def pruefe_loesung(aufruf):
    werkzeug = aufruf["tool_name"]
    eingabe = aufruf["tool_input"]

    if werkzeug in ("Read", "Edit", "Write"):
        if PurePosixPath(eingabe["file_path"]).name == ".env":
            return "blockieren", "Die .env enthält Secrets. Vorlage: .env.example"

    if werkzeug == "Bash":
        if ENV_IM_BEFEHL.search(eingabe["command"]):
            return "blockieren", "Der Befehl greift auf die .env zu. Sie enthält Secrets. Vorlage: .env.example"

    return "erlauben", ""


teste(pruefe_loesung, FAELLE_ENV)

       Nr  Soll        Ist         Tool   Aufruf
        1  blockieren  blockieren  Read   .env
        2  erlauben    erlauben    Read   .env.example
        3  erlauben    erlauben    Read   README.md
        4  erlauben    erlauben    Edit   .env.example
        5  blockieren  blockieren  Write  .env
        6  blockieren  blockieren  Bash   cat .env
        7  erlauben    erlauben    Bash   cat .env.example
        8  blockieren  blockieren  Bash   echo "DB_PASSWORD=geheim" >> .env
        9  blockieren  blockieren  Bash   cp .env.example .env
       10  blockieren  blockieren  Bash   grep TOKEN .env
       11  blockieren  blockieren  Bash   cp .env /tmp/env-kopie
       12  blockieren  blockieren  Bash   python3 -c "print(open('.env').read())"
       13  erlauben    erlauben    Bash   mvn spring-boot:run

✓ 13 von 13 wie erwartet.


Zwei Dinge fallen auf:

- Nr. 9, `cp .env.example .env`, ist der Schritt aus der README des Dienstes. Der Hook blockiert ihn trotzdem, weil er die `.env` schreibt. Das ist der Preis einer harten Regel: Diesen Schritt macht ab jetzt ein Mensch.
- Der Hook vergleicht Text. Er weiss nicht, was `cat`, `cp` oder `python3` tun, er sieht nur, ob `.env` im Befehl steht. Wie die Bash-Regeln bei den Permissions.

## 3. Bonus: Umwege suchen und schliessen

Jetzt seid ihr der Agent und wollt an die `.env`.

> **Aufgabe 4** (Bonus): Baut Aufrufe mit `bash("…")`, `read("…")`, `write("pfad", "inhalt")` oder `grep("muster", "pfad")` und schaut, was `pruefe` dazu sagt. Gesucht ist ein Aufruf, der die `.env` liest oder schreibt und trotzdem „erlauben“ bekommt.

<details><summary>Tipp</summary>

Was macht die Shell mit dem Befehl, bevor `cat` ihn bekommt? Anführungszeichen, `*` und `?` sind Kandidaten. Und welche Tools kommen im Test oben gar nicht vor?

</details>

In [4]:
probiere(
    pruefe,
    bash("cat .env"),
    bash("head -n 1 .env"),
)

erlauben    Bash   cat .env
erlauben    Bash   head -n 1 .env


In [5]:
# @title Lösung (aufklappen)
umwege = [
    (bash('cat .e""nv'), "blockieren"),                                # die Shell setzt .e""nv zu .env zusammen
    (bash("cat .en?"), "blockieren"),                                  # ? ist ein Platzhalter, die Shell findet .env
    (bash("echo Y2F0IC5lbnY= | base64 -d | sh"), "blockieren"),        # „cat .env“ in Base64
    (write("lesen.py", "print(open('.env').read())"), "erlauben"),     # ein Skript schreiben ist harmlos …
    (bash("python3 lesen.py"), "blockieren"),                          # … es auszuführen nicht
    (grep(".", ".env"), "blockieren"),                                 # ein Tool, an das niemand gedacht hat
]
teste(pruefe_loesung, umwege)

       Nr  Soll        Ist         Tool   Aufruf
FALSCH  1  blockieren  erlauben    Bash   cat .e""nv
FALSCH  2  blockieren  erlauben    Bash   cat .en?
FALSCH  3  blockieren  erlauben    Bash   echo Y2F0IC5lbnY= | base64 -d | sh
        4  erlauben    erlauben    Write  lesen.py
FALSCH  5  blockieren  erlauben    Bash   python3 lesen.py
FALSCH  6  blockieren  erlauben    Grep   . in .env

1 von 6 wie erwartet. Falsch: 1, 2, 3, 5, 6


Jede dieser Lücken lässt sich mit einer weiteren Regel schliessen. Zwei davon sind die Knobelaufgabe: das Grep-Tool und Anführungszeichen, die die Shell entfernt.

> **Aufgabe 5** (Bonus, Knobelaufgabe): `pruefe_dicht` so erweitern, dass sie `grep("TOKEN", ".env")` und `cat .e""nv` blockiert, ohne dass einer der 13 Fälle von oben kippt. Der Test `FAELLE_UMWEGE` enthält die 13 Fälle und fünf neue, darunter zwei harmlose, die erlaubt bleiben müssen. Mit eurer Funktion aus Aufgabe 3 als Startpunkt stimmen 15 von 18. Fertig bei „✓ 18 von 18 wie erwartet.“

In [6]:
def pruefe_dicht(aufruf):
    # Startpunkt ist eure Funktion aus Aufgabe 3. Schafft sie die 13 noch nicht: pruefe durch pruefe_loesung ersetzen.
    return pruefe(aufruf)


teste(pruefe_dicht, FAELLE_UMWEGE)

       Nr  Soll        Ist         Tool   Aufruf
        1  blockieren  blockieren  Read   .env
        2  erlauben    erlauben    Read   .env.example
        3  erlauben    erlauben    Read   README.md
        4  erlauben    erlauben    Edit   .env.example
        5  blockieren  blockieren  Write  .env
FALSCH  6  blockieren  erlauben    Bash   cat .env
        7  erlauben    erlauben    Bash   cat .env.example
FALSCH  8  blockieren  erlauben    Bash   echo "DB_PASSWORD=geheim" >> .env
FALSCH  9  blockieren  erlauben    Bash   cp .env.example .env
FALSCH 10  blockieren  erlauben    Bash   grep TOKEN .env
FALSCH 11  blockieren  erlauben    Bash   cp .env /tmp/env-kopie
FALSCH 12  blockieren  erlauben    Bash   python3 -c "print(open('.env').read())"
       13  erlauben    erlauben    Bash   mvn spring-boot:run
FALSCH 14  blockieren  erlauben    Grep   TOKEN in .env
       15  erlauben    erlauben    Grep   DB_PASSWORD in .env.example
       16  erlauben    erlauben    Grep   TODO in src

<details><summary>Tipp 1</summary>

Grep: In welchem Feld steht der Pfad? `print(grep("TOKEN", ".env"))` zeigt es. Shell: Was bleibt von `cat .e""nv` und `cat '.e'nv` übrig, wenn man die Anführungszeichen weglässt?

</details>
<details><summary>Tipp 2</summary>

Neu prüfen muss man nichts. Es reicht, den Aufruf umzuformen und an `pruefe` weiterzugeben:

```python
    if aufruf["tool_name"] == "Grep":
        return pruefe(read(…))   # den Pfad aus aufruf["tool_input"] einsetzen
    if aufruf["tool_name"] == "Bash":
        befehl = aufruf["tool_input"]["command"].replace('"', "")   # dasselbe noch für '
        return pruefe(bash(befehl))
```

</details>

In [7]:
# @title Lösung (aufklappen)
def pruefe_dicht_loesung(aufruf):
    werkzeug = aufruf["tool_name"]
    eingabe = aufruf["tool_input"]

    # Grep hat den Pfad in path, nicht in file_path. Also so prüfen, als würde die Datei gelesen.
    if werkzeug == "Grep":
        return pruefe_loesung(read(eingabe.get("path", "")))

    # Die Shell entfernt Anführungszeichen, bevor cat den Namen sieht: Aus .e""nv und '.e'nv wird .env.
    # Also den Befehl ohne Anführungszeichen prüfen. .env.example bleibt dabei erlaubt.
    if werkzeug == "Bash":
        ohne = eingabe["command"].replace('"', "").replace("'", "")
        return pruefe_loesung(bash(ohne))

    return pruefe_loesung(aufruf)


teste(pruefe_dicht_loesung, FAELLE_UMWEGE)

       Nr  Soll        Ist         Tool   Aufruf
        1  blockieren  blockieren  Read   .env
        2  erlauben    erlauben    Read   .env.example
        3  erlauben    erlauben    Read   README.md
        4  erlauben    erlauben    Edit   .env.example
        5  blockieren  blockieren  Write  .env
        6  blockieren  blockieren  Bash   cat .env
        7  erlauben    erlauben    Bash   cat .env.example
        8  blockieren  blockieren  Bash   echo "DB_PASSWORD=geheim" >> .env
        9  blockieren  blockieren  Bash   cp .env.example .env
       10  blockieren  blockieren  Bash   grep TOKEN .env
       11  blockieren  blockieren  Bash   cp .env /tmp/env-kopie
       12  blockieren  blockieren  Bash   python3 -c "print(open('.env').read())"
       13  erlauben    erlauben    Bash   mvn spring-boot:run
       14  blockieren  blockieren  Grep   TOKEN in .env
       15  erlauben    erlauben    Grep   DB_PASSWORD in .env.example
       16  erlauben    erlauben    Grep   TODO in src

Damit sind zwei Umwege zu, die anderen aus Aufgabe 4 bleiben offen: `cat .en?`, Base64 und das Skript, das die Datei liest. Die Shell hat mehr Schreibweisen, als ein Hook Regeln hat. Und in `python3 lesen.py` steht gar nichts von der `.env`: Der Hook prüft den Aufruf, nicht das Programm.

Für den echten Hook kommt noch etwas dazu: Damit er Grep überhaupt zu sehen bekommt, muss Grep im `matcher` stehen (Abschnitt 4).

## 4. Aus der Funktion einen echten Hook machen

Zum Ansehen. Die Funktion kommt in eine Datei, dazu ein paar Zeilen, die den Aufruf von stdin lesen und das Ergebnis als Exit-Code melden. Die fertige Datei liegt im Ordner als `env_hook.py`.

```python
aufruf = json.load(sys.stdin)        # Claude Code schreibt den Aufruf als JSON auf stdin
entscheid, grund = pruefe(aufruf)
if entscheid == "blockieren":
    print(grund, file=sys.stderr)    # stderr geht als Begründung an Claude
    sys.exit(2)                      # Exit-Code 2 blockiert den Aufruf
sys.exit(0)                          # 0: weiter
```

Dazu der Eintrag in `.claude/settings.json` des Projekts, aufgebaut wie bei AstroPlanner:

```json
{
  "hooks": {
    "PreToolUse": [{
      "matcher": "Read|Edit|Write|Bash",
      "hooks": [{
        "type": "command",
        "command": "python3 \"$CLAUDE_PROJECT_DIR/.claude/hooks/env_hook.py\""
      }]
    }]
  }
}
```

Was laut Doku von Claude Code dazugehört:

- Der `matcher` ist ein regulärer Ausdruck über den Tool-Namen. Für andere Tools läuft der Hook gar nicht, hier etwa für Grep.
- Nur Exit-Code 2 blockiert. Stürzt der Hook ab, endet er mit einem anderen Code oder läuft er in den Timeout, geht der Aufruf durch.
- Statt mit Exit-Code 2 kann ein Hook auch mit JSON antworten (`"permissionDecision": "deny"`). Für diese Übung reicht der Exit-Code.

Die Zelle unten ruft `env_hook.py` so auf, wie Claude Code es tut, mit dem Aufruf als JSON auf stdin. Ausführen ist freiwillig.

In [8]:
import subprocess
import sys

for aufruf in [read(".env"), bash("cat .env.example")]:
    lauf = subprocess.run([sys.executable, "env_hook.py"], input=json.dumps(aufruf), capture_output=True, text=True)
    print(f"{aufruf['tool_name']} {kurz(aufruf)}: Exit-Code {lauf.returncode} {lauf.stderr.strip()}")

Read .env: Exit-Code 2 Die .env enthält Secrets. Vorlage: .env.example
Bash cat .env.example: Exit-Code 0 


## Was ihr gebaut habt

Ein Hook ist eine Funktion über dem Aufruf: Tool und Eingabe hinein, „erlauben“ oder „blockieren“ mit Grund heraus. Eure Funktion sperrt die `.env` für Read, Edit, Write und die Shell, soweit `.env` im Befehl steht. Sie vergleicht Text und weiss nicht, was ein Programm tut; darauf kommen wir am Nachmittag bei Security zurück.